# exp009 — reproduction of the official ink_9um training recipe (20,000 of 78,125 iterations) on Kaggle

**Settings:** Accelerator **GPU T4 x2** (1 GPU used), Internet **on**, notebook **private**, inputs: the private outputs of the two
exp008 corpus notebooks. Config: `configs/exp009_repro_train.yaml`. Official recipe, official LR curve (`max_steps` 78,125), stopped at 20,000;
deviations are listed in the config header. Expected ~7-8 h (loader-bound, ~50 examples/s). Heavy files in `/tmp/ks`; only checkpoints
and small result files go to `/kaggle/working/outputs/exp009_repro_train/`. Every cell ends successfully so the notebook always saves its output.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
echo "cpus: $(nproc)"; free -g | sed -n 2p; df -h /kaggle/working | tail -1; python --version
echo "--- inputs"; ls /kaggle/input 2>&1 | head
mkdir -p /tmp/ks
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /tmp/ks/harness
git -C /tmp/ks/harness log -1 --format='harness %H'
true

In [ ]:
import os, glob, yaml, pathlib
SCR = "/tmp/ks"
H = pathlib.Path(SCR + "/harness")
cfg = yaml.safe_load((H / "configs/exp009_repro_train.yaml").read_text())
seg = cfg["segments_dir"]
roots = {}
for group, probe in (("A", "pherc0139-w016"), ("B", "phercparis4-w00")):
    hits = sorted(glob.glob(f"/kaggle/input/**/{seg}/{probe}", recursive=True))
    roots[group] = hits[0].split("/" + seg)[0] if hits else ""
    print(group, "->", roots[group] or "NOT FOUND")
os.environ.update({
    "H": str(H), "CFG": str(H / "configs/exp009_repro_train.yaml"),
    "VILLA_SHA": cfg["upstream"]["villa_commit"], "PY_VER": cfg["upstream"]["python"],
    "UV_SYNC_ARGS": " ".join(cfg["upstream"]["uv_sync_args"]),
    "VES": SCR + "/villa/vesuvius", "VILLA": SCR + "/villa",
    "CORPUS_A": roots["A"], "CORPUS_B": roots["B"],
    "RUN": SCR + "/exp009/run", "SMOKE": SCR + "/exp009/smoke", "WORK": SCR + "/exp009",
    "RESULTS": "/kaggle/working/outputs/exp009_repro_train", "TIMEOUT": str(cfg["timeout_seconds"]),
})

## 1. Upstream code (villa, pinned commit) + environment

In [ ]:
%%bash
set -e
git init -q /tmp/ks/villa && cd /tmp/ks/villa
git remote add origin https://github.com/ScrollPrize/villa.git
git sparse-checkout set vesuvius volume-cartographer segmentation/models/batchgeneratorsv2 segmentation/models/arch/nnunet
git fetch -q --depth 1 --filter=blob:none origin "$VILLA_SHA"
git checkout -q FETCH_HEAD && git log -1 --format='villa %H %cd'

In [ ]:
%%bash
set -e
pip install -q uv==0.11.24
uv python install "$PY_VER"
cd "$VES" && uv sync --python "$PY_VER" $UV_SYNC_ARGS 2>&1 | tail -3
uv run --no-sync python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"

## 2. Data terms
Already accepted by the owner for this account in exp000. Nothing here runs `vesuvius.accept_terms`.

## 3. Mounted corpus check and training configs (smoke run + long run), validated with villa's TrainingConfig

In [ ]:
%%bash
mkdir -p "$WORK" "$RESULTS"
echo "corpus A: $CORPUS_A"; echo "corpus B: $CORPUS_B"
for c in "$CORPUS_A" "$CORPUS_B"; do [ -n "$c" ] && python3 -c "
import json, sys
r = json.load(open('$c/corpus.json')); print('corpus', r['group'], '| ok', len(r['segments']), '| FAILED', [f['key'] for f in r['failed']], '| fetched GB', round(r['fetch_stats']['bytes']/1e9, 2))"; done
cd "$VES"
uv run --no-sync python "$H/scripts/exp009_make_train_config.py" --config "$CFG" --villa "$VILLA" --corpus-a "$CORPUS_A" --corpus-b "$CORPUS_B" --out "$WORK/smoke.json" --out-dir "$SMOKE" --smoke 2>&1 | tail -30
uv run --no-sync python "$H/scripts/exp009_make_train_config.py" --config "$CFG" --villa "$VILLA" --corpus-a "$CORPUS_A" --corpus-b "$CORPUS_B" --out "$WORK/long.json" --out-dir "$RUN" 2>&1 | tail -3
ls -la "$WORK"
cp "$WORK"/smoke.json "$WORK"/long.json "$RESULTS/" 2>/dev/null
true

## 4. Smoke run (120 steps on the real corpus). The long run starts only if it produced a checkpoint.

In [ ]:
%%bash
cd "$VES"
timeout 1800 uv run --no-sync python -m vesuvius.ink_detection.training.train "$WORK/smoke.json" > "$WORK/smoke.log" 2>&1 < /dev/null
echo "smoke exit status: $?"
ls "$SMOKE" 2>/dev/null | head; echo "---- last lines"; tr '\r' '\n' < "$WORK/smoke.log" | grep -E "sampling_audit|Error|Traceback|loss=" | tail -4 | cut -c1-260
if ls "$SMOKE"/*.pth >/dev/null 2>&1; then echo SMOKE_OK > "$WORK/smoke_ok"; else echo "no smoke checkpoint"; fi
cp "$WORK/smoke.log" "$RESULTS/" 2>/dev/null
true

## 5. Long run: 20,000 iterations at the official LR schedule (capped at 9.5 h; checkpoints every 5,000)

In [ ]:
%%bash
cd "$VES"
if [ -f "$WORK/smoke_ok" ]; then
  echo "long run start $(date +%T)"
  timeout "$TIMEOUT" uv run --no-sync python -m vesuvius.ink_detection.training.train "$WORK/long.json" > "$WORK/long.log" 2>&1 < /dev/null
  echo "long run exit status: $? at $(date +%T)"
else
  echo "SKIPPED: the smoke run did not produce a checkpoint (see smoke.log in the output)"
fi
true

## 6. Results kept as notebook output: checkpoints, validation metrics, logs (no preview images)

In [ ]:
%%bash
mkdir -p "$RESULTS/run"
cp "$RUN"/*.pth "$RESULTS/run/" 2>/dev/null
cp "$RUN"/*.json "$RUN"/*.jsonl "$RESULTS/run/" 2>/dev/null
cp "$WORK/long.log" "$RESULTS/" 2>/dev/null
echo "=== validation metrics ==="; cat "$RUN"/validation_metrics.jsonl 2>/dev/null | cut -c1-300
echo "=== last progress lines ==="; tr '\r' '\n' < "$WORK/long.log" 2>/dev/null | grep -E "loss=" | tail -3 | cut -c1-200
echo "=== files kept ==="; (cd "$RESULTS" && find . -type f | sort | xargs ls -la | awk '{print $5, $9}'); du -sh "$RESULTS"
true